In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv
from sklearn.preprocessing import MinMaxScaler
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.utils.parametrizations as parametrizations
import torch.optim as optim
import time
from sklearn.metrics import mean_squared_error, mean_absolute_error
import torch.fft
import json
import joblib
import matplotlib.pyplot as plt
from datetime import datetime

# --- .env file betöltése --- Ebben tárolom az adatbáziskapcsolathoz szükséges adatokat.
load_dotenv()

# --- Adatbázis Kapcsolat --- Az előlre letöltött adatok letöltése.
engine = create_engine(f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}")

# --- Lekérdezés --- A era5_adatok_fix táblából letöltök minden adatot.
query = "SELECT valid_time, temp_c, dewpoint_c, rh, wind_speed, wind_dir, sp_hpa, tp, skt_c, ssrd_wm2, fg10_kmh FROM era5_adatok_fix ORDER BY valid_time"
df = pd.read_sql(query, engine, parse_dates=["valid_time"])

# --- Időbélyeg kódolása (Ciklikus jellemzők) ---
# Ez segít a TCN-nek a napi/éves ciklusok felismerésében.
def encode_cyclic_time(df, col):
    # Óra kódolása (0-23, tehát 24 egység!)
    # Megjegyzés: itt érdemesebb 24-gyel osztani, mert 0-tól 23-ig 24 állapot van
    #LINEÁRIS IDŐ: Unix timestamp (másodpercek), ez azért fontos, hogy az évek közötti változást is tudja értelmezni majd a model.
    df = df.sort_values(col).reset_index(drop=True)
    df['time_linear'] = df[col].astype('int64') // 10**9

    #Napok tördelése, sin és cos ciklusba tördelem így reprezentál egy napot.
    hour = df[col].dt.hour
    df['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    
    # Év napja (1-365 vagy 1-366)
    day_of_year = df[col].dt.dayofyear
    
    # Megnézzük, szökőév-e az adott sor dátuma
    # Ha szökőév, 366-tal osztunk, ha nem, 365-tel
    days_in_year = np.where(df[col].dt.is_leap_year, 366, 365)

    #Hasonlóan a napokhoz az éveket is körkörös sin és cos ciklusokba tördelem
    df['year_sin'] = np.sin(2 * np.pi * (day_of_year - 1) / days_in_year)
    df['year_cos'] = np.cos(2 * np.pi * (day_of_year - 1) / days_in_year)

    # A kódod elején az encode_cyclic_time mintájára:
    df['wind_dir_sin'] = np.sin(2 * np.pi * df['wind_dir'] / 360.0)
    df['wind_dir_cos'] = np.cos(2 * np.pi * df['wind_dir'] / 360.0)
    
    return df

#Ezzel kész az adattömb minden szükséges plusz generált oszloppal.
df = encode_cyclic_time(df, "valid_time")



In [2]:
# DEF: parse_list_cell
# Listába tördeli egy vesszővel felsorolt szöveg elemeit.
# Ez kell, hogy a filokból a feature és más felsorolási elemeket darabokra bontsam a használathoz.
def parse_list_cell(cell):
    cell = str(cell).strip()
    if not cell:
        return []
    return [c.strip() for c in cell.split(",")]

# Eszköz kiválasztása: ha van GPU, azt használjuk, egyébként a CPU-t
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"A kód jelenleg ezt használja: {device}")
if device.type == 'cuda':
    print(f"GPU típusa: {torch.cuda.get_device_name(0)}")

A kód jelenleg ezt használja: cuda
GPU típusa: Tesla T4


In [3]:
class GroupScaler:
    def __init__(self, groups=None):
        """
        groups: dict, pl. {"celsius": ["temp_c", "dewpoint_c", "skt_c"]}
        """
        self.groups = groups if groups else {}
        self.group_scalers = {}
        self.standalone_scaler = MinMaxScaler()
        self.all_features = []
        self.standalone_features = []

    def fit(self, df, features):
        self.all_features = features
        grouped_cols = [col for cols in self.groups.values() for col in cols if col in features]
        self.standalone_features = [f for f in features if f not in grouped_cols]

        # Csoportos scalert tanítunk
        for name, cols in self.groups.items():
            active_cols = [c for c in cols if c in features]
            if active_cols:
                # Összegyúrjuk az oszlopokat egy hosszú oszloppá, hogy közös min/max legyen
                combined_data = df[active_cols].values.reshape(-1, 1)
                scaler = MinMaxScaler().fit(combined_data)
                self.group_scalers[name] = scaler

        # Maradék oszlopok scalert tanítunk
        if self.standalone_features:
            self.standalone_scaler.fit(df[self.standalone_features])
        return self

    def transform(self, df):
        df_out = df.copy()
        # Csoportos transzformáció
        for name, cols in self.groups.items():
            active_cols = [c for c in cols if c in self.all_features]
            if active_cols:
                scaler = self.group_scalers[name]
                for col in active_cols:
                    df_out[col] = scaler.transform(df_out[[col]].values)
        
        # Standalone transzformáció
        if self.standalone_features:
            df_out[self.standalone_features] = self.standalone_scaler.transform(df_out[self.standalone_features])
        return df_out

    def inverse_transform_single_col(self, data_array, col_name):
        """Egyetlen oszlop (vagy predikciós tömb) visszaskálázása"""
        # Megnézzük, melyik csoporthoz tartozik
        for name, cols in self.groups.items():
            if col_name in cols:
                scaler = self.group_scalers[name]
                return scaler.inverse_transform(data_array.reshape(-1, 1)).reshape(data_array.shape)
        
        # Ha standalone
        if col_name in self.standalone_features:
            # Itt trükkösebb, mert a MinMaxScaler több oszlopot vár. 
            # Megkeressük az indexét a standalone scalerben.
            idx = self.standalone_features.index(col_name)
            dummy = np.zeros((len(data_array.flatten()), len(self.standalone_features)))
            dummy[:, idx] = data_array.flatten()
            inv_dummy = self.standalone_scaler.inverse_transform(dummy)
            return inv_dummy[:, idx].reshape(data_array.shape)
        
        return data_array # Ha nincs meg, ne változtasson

# Deffiniálom az azonos mértékegységű csoportokat.
SCALING_GROUPS = {
    "celsius": ["temp_c", "dewpoint_c", "skt_c"],
    "wind": ["wind_speed", "fg10_kmh"]
}

class ManualGroupScaler:
    def __init__(self, feature_limits):
        """
        feature_limits: dict, pl. {"temp_c": (-30.0, 45.0), "rh": (0.0, 100.0)}
        """
        self.limits = feature_limits
        self.all_features = []

    def fit(self, df, features):
        # Erre a függvényre a kompatibilitás miatt van szükség, de nem tanul semmit az adatokból!
        self.all_features = features
        return self

    def transform(self, df):
        df_out = df.copy()
        for col in self.all_features:
            if col in self.limits:
                f_min, f_max = self.limits[col]
                # Sima Min-Max skálázás képlettel: (x - min) / (max - min)
                df_out[col] = (df_out[col] - f_min) / (f_max - f_min)
            # Aminek nincs határa (pl. sin/cos oszlopok), azt békén hagyja, mert az eleve -1 és 1 között van
        return df_out

    def inverse_transform_single_col(self, data_array, col_name):
        """Visszaskálázás a valós fizikai dimenzióba"""
        if col_name in self.limits:
            f_min, f_max = self.limits[col_name]
            # Inverz képlet: x * (max - min) + min
            return data_array * (f_max - f_min) + f_min
        return data_array

# Határozzuk meg a fizikai korlátokat a te régiódra szabva
WEATHER_LIMITS = {
    "temp_c": (-100.0, 100.0),
    "dewpoint_c": (-100.0, +100.0),
    "skt_c": (-100.0, 100.0),       # Talajfelszín hőmérséklet kicsit tágabb
    "rh": (0.0, 100.0),
    "wind_speed": (0.0, 600.0),    # km/h vagy m/s szerint (igazítsd a saját adatodhoz!)
    "fg10_kmh": (0.0, 600.0),
    "wind_dir": (0.0, 360.0),
    "sp_hpa": (800.0, 1100.0),     # Légnyomás hPa
    "tp": (0.0, 400.0),             # Csapadék max mm
    "ssrd_wm2": (0.0, 2000.0)      # Globálsugárzás max
}

In [4]:
# DEF: run_experiments
# Ez az automata tesztelési folyamat indítása, megadható a file amiből elindul a folyamat és az almappa amibe a mentést végzi. 
def run_experiments(config_path, base_output_dir="experiments"):
    os.makedirs(base_output_dir, exist_ok=True)
    config = pd.read_csv(config_path, sep="\t")  # vagy sep="," ha CSV, jupyterben könnyebb használni a TSV formátumot.
    all_results = []
# A file soronként történő feldolgozása
    for idx, row in config.iterrows():
        test_name = row["test_name"]
        features = parse_list_cell(row["features"])
        target_col = parse_list_cell(row["target_col"])
        extra_cols = parse_list_cell(row.get("extra_cols", ""))
        input_window = int(row["input_window"])
        output_horizon = int(row["output_horizon"])
        mask_prob = float(row["mask_prob"])
        epoch = int(row["epoch"])

        print(f"\n=== Futtatás: {test_name} ===")
        # Tesztenként külön almappa készítése
        out_dir = os.path.join(base_output_dir, test_name)
        os.makedirs(out_dir, exist_ok=True)
        # Ciklikusan meghívom az egy tesztre vonatkozó folyamatot
        result = run_single_experiment(
            test_name=test_name,
            features=features,
            target_col=target_col,
            extra_cols=extra_cols,
            input_window=input_window,
            output_horizon=output_horizon,
            mask_prob=mask_prob, # Ez fontos jelenleg hogy 0-1 (reprezentálja 0-100%) érték között legyen megadva a fileba.
            out_dir=out_dir,
            epochs=epoch
        )

        all_results.append(result)
    # Összegző riport készítése
    generate_master_html_summary(all_results, base_output_dir)

# DEF: run_single_experiment
# Egy adott teszt lefuttatása, itt már átadva minden tesztfeltétel és mentési hely.
def run_single_experiment(test_name, features, target_col, extra_cols,
                          input_window, output_horizon, mask_prob, out_dir, epochs):
    start_time_all = time.time() #Idő mérése egy teljes teszt lefutására
    df_local = df.copy() #Bemásolom a globális változóból az adattömböt

    # ÚJ SKÁLÁZÁS:
    #scaler = GroupScaler(groups=SCALING_GROUPS)
    scaler = ManualGroupScaler(feature_limits=WEATHER_LIMITS)
    scaler.fit(df_local, features)
    df_local = scaler.transform(df_local)
    
    # --- Adattisztítás ---
    df_local = df_local.ffill().bfill()
    
    # Konvertálás float32-re a PyTorch-hoz
    data_array = df_local[features].values.astype(np.float32)
    
    print(f"Adatok betöltve: {data_array.shape[0]} sor, {data_array.shape[1]} jellemző.")
    
    #------Indexek amik targetek és amik ne legyenek hiányosra generálva közben.------
    # Target index automatikusan generálva a file alapján.
    target_index = [features.index(col) for col in target_col]
    
    # NaN és üres értékek kiszűrése (Pandas/TSV beolvasás miatt)
    if isinstance(extra_cols, list):
        # Csak azokat tartjuk meg, amik stringek és nem 'nan' (vagy nem üresek)
        extra_cols = [col for col in extra_cols if isinstance(col, str) and col.lower() != 'nan' and col.strip() != '']
    else:
        # Ha eleve nem lista (pl. egyetlen NaN érték), legyen üres lista
        extra_cols = []
    
    # Extra feature-ök amik nem lesznek hiányosra generálva pluszba azokon felül amik célok.
    # Ezek amik sosem lehetnek hiányosak alaphelyzetben a célokon felül.
    #default_extra = ["hour_sin", "hour_cos", "year_sin", "year_cos", "time_linear"]
    default_extra = ["hour_sin", "hour_cos", "year_sin", "year_cos","wind_dir_sin","wind_dir_cos"]
    # összevonás duplikáció nélkül, amiket pluszban megadunk a fileban itt adódik hozzá.
    extra_cols = list(dict.fromkeys(extra_cols + default_extra))
    
    # Biztonságos index keresés (Csak ha az oszlop tényleg benne van a features-ben)
    keep_indices = target_index + [features.index(col) for col in extra_cols if col in features]    

    # Datasetek létrehozása, ebből lesz a train és test csomag
    full_dataset = TimeSeriesDataset(
        data_array, 
        input_window, 
        output_horizon, 
        target_idx=target_index,
        mask_prob=mask_prob,
        always_keep=keep_indices
    )
    
    # Split (10% tesztadat elég nagy mennyiség figyelembe véve az adathalmazt, 8év,lesz teszt amibe 1 év teszt lesz csak, 0.98 -> 2%)
    train_size = int(len(full_dataset) * 0.98)

    #Ranodm vágással is próbáltam, de sokat ront a kontinuitáson így rosszabb eredményeket hoz ebben az esetben.
    #test_size = len(full_dataset) - train_size
    #train_dataset, test_dataset = torch.utils.data.random_split(full_dataset, [train_size, test_size])

    #Az adathalaz végéről vágom le a test-et. Így mindig az utosó ~1 éven teszteli. 
    train_dataset = torch.utils.data.Subset(full_dataset, range(0, train_size))
    test_dataset = torch.utils.data.Subset(full_dataset, range(train_size, len(full_dataset)))
    
    # DataLoader - Ez küldi az adatokat a GPU-nak batch-ekben
    # Talán majd érdemes lehet megpróbálni tone-olni a batch méretet: A tesztek alapján 64 jó mert így teljesen ki van terhelve a GPU.
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True, persistent_workers=True)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)
    
    print(f"Tanító batch-ek száma: {len(train_loader)}")


    # Modellek inicializálása
    num_targets = len(target_index) 
    model_tcn = TCNModel( num_inputs=len(features), output_horizon=output_horizon, num_targets=num_targets ).to(device)

    # TCN Tanítás
    criterion = nn.MSELoss() # A hiba mérése
    optimizer = optim.Adam(model_tcn.parameters(), lr=0.001) # Az algoritmus, ami állítja a súlyokat
    #epochs = 10 # Kezdésnek 10 kör, már file változóból érkezik.
    # AMP scaler 
    #amp_scaler = torch.amp.GradScaler("cuda")
    
    print(f"Tanítás megkezdése a(z) {device} eszközön...")
    tcn_epoch_logs = []
    for epoch in range(epochs):
        start_time = time.time()  #Egy epoch idejének mérése
        model_tcn.train() # Modell tanítási módba állítása
        running_loss = 0.0
        
        for batch_idx, (batch_x, batch_y) in enumerate(train_loader):
            # Adatok átküldése a GPU-ra
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            # Gradiens törlése (minden lépésnél nullázás)
            optimizer.zero_grad()
            
            """
            # Mixed precision forward pass (Kikapcsoltam mert timesnetnél instabil, inkább itt sem használom...)
            with torch.amp.autocast("cuda"):
                outputs = model_tcn(batch_x)  #Előrejelzés
                loss = criterion(outputs, batch_y) #Hibák kiszámítása

            # Backward pass AMP-vel 
            amp_scaler.scale(loss).backward() # Hibavisszaterjesztés, ezzel tanul a hibáiból a modell.
            amp_scaler.step(optimizer)
            amp_scaler.update()
            """
            
            outputs = model_tcn(batch_x)  #Előrejelzés
            loss = criterion(outputs, batch_y) #Hibák kiszámítása
            loss.backward() #AMP nélkül
            # Súlyok frissítése
            optimizer.step() #AMP nélkül

            #A loss értékek összeadása a végén a teljes számításhoz egy epoch-n.
            running_loss += loss.item()
            
            # 400 batch-enként kiírjuk, hol tartunk, hogy ne legyen csendben
            if batch_idx % 400 == 0:
                print(f"TCN - Epoch {epoch+1} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.6f}")
        #Itt osztódik el átlag veszteségre.
        epoch_loss = running_loss / len(train_loader)
        #Egy epoch teljes idejének számítása
        duration = time.time() - start_time
        print(f"--- TCN Epoch [{epoch+1}/{epochs}] kész. Átlagos hiba: {epoch_loss:.6f} | Idő: {duration:.2f}s ---")
        #Epoch ciklus gyűjtése a végső log-hoz.
        tcn_epoch_logs.append({"epoch": epoch + 1, "loss": epoch_loss, "duration": duration})


    # Kiértékelés
    # Modell átállítása értékelési módba és adatok GPU-ra küldése
    #TCN kiértékelése
    model_tcn.eval() 
    all_preds = []
    all_actuals = []
    
    with torch.no_grad(): # Kikapcsolom a gradiens számítást (gyorsabb, kevesebb memória)
        for batch_x, batch_y in test_loader:
            batch_x = batch_x.to(device)
            
            # Predikció
            outputs = model_tcn(batch_x)
            
            all_preds.append(outputs.cpu().numpy()) #A predikciók felgyűjtése
            all_actuals.append(batch_y.numpy()) #Az aktuális tesztérték kiolvasása az adott predikció mellé.
    
    # Tömbök összefűzése
    y_pred = np.concatenate(all_preds, axis=0)
    y_test = np.concatenate(all_actuals, axis=0)

    #Visszaskálázása az eredményeknek a normál értékekre a scaler alapján.
    y_pred_real, y_test_real = inverse_transform_multitarget(y_pred, y_test, scaler, target_index, len(features))

    print("TCN predikciók kész és visszaskálázva.")

    # Metrikák kiszámítása
    #TCN
    tcn_overall = calculate_metrics_overall_fixed(y_test_real, y_pred_real, "TCN")
    tcn_per_target = calculate_metrics_per_target(y_test_real, y_pred_real, target_col)
    tcn_mae_h, tcn_rmse_h = calculate_metrics_per_horizon(y_test_real, y_pred_real)
    tcn_horizon_metrics = calculate_metrics_per_horizon_per_target(y_test_real, y_pred_real, target_col)
    
    # Metrikák mentése JSON-ba
    json.dump({
        "tcn_overall": tcn_overall,
        "tcn_per_target": tcn_per_target,
    }, open(os.path.join(out_dir, "metrics_summary.json"), "w"), indent=2)

    # Grafikonok mentése PNG-be
    save_plots(y_test_real,y_pred_real,target_col,out_dir,test_name)

    # Modellek + scaler + paraméterek mentése
    save_models_and_params(out_dir, model_tcn, scaler,
                           features, target_col, extra_cols,
                           input_window, output_horizon, mask_prob)

    # HTML riport generálása
    generate_html_report(out_dir, test_name,
                         tcn_overall, tcn_per_target,
                         features, target_col, extra_cols, input_window, output_horizon, mask_prob,
                         tcn_epoch_logs, epochs, 
                         tcn_mae_h, tcn_rmse_h, tcn_horizon_metrics)
    
    #A teljes teszt idejének számítása
    end_time = time.time() 
    runtime_seconds = end_time - start_time_all
    #Visszatérünk az összes fő értékkel ami fontos a központi riporthoz.
    return {
        "test_name": test_name,
        "features": ",".join(features),
        "target_col": ",".join(target_col),
        "extra_cols": ",".join(extra_cols),
        "input_window": input_window,
        "output_horizon": output_horizon,
        "mask_prob": mask_prob,
        "epochs": epochs,
        "tcn_mse": tcn_overall["mse"],
        "tcn_mae": tcn_overall["mae"],
        "tcn_rmse": tcn_overall["rmse"],
        "runtime_seconds": runtime_seconds
    }

#DEF: generate_master_html_summary
#Ezzel gánárálom a teljes összegző riportot
#Az összefűzött adatokat várja és a kimeneti mappát.
def generate_master_html_summary(all_results, base_output_dir,
                                 researcher_name="Bagi Béla", 
                                 institution="University of Miskolc - Institute of Physics and Electrical Engineering",
                                 project_name = "Home Weather Station Prediction",
                                 version="1.0"):
    
    now = datetime.now().strftime("%Y-%m-%d %H:%M")
    
    rows = []
    for r in all_results:
        # Meghatározzuk melyik modell volt a jobb (zölddel kiemeljük)
        
        rows.append(f"""
        <tr>
            <td style="text-align:left; font-weight:600;">{r['test_name']}</td>
            <td style="font-size: 11px; text-align:left; max-width: 250px; overflow: hidden; text-overflow: ellipsis;">{r['features']}</td>
            <td>{r['target_col']}</td>
            <td>{r['extra_cols']}</td>
            <td>{r['input_window']}</td>
            <td>{r['output_horizon']}</td>
            <td>{r.get('mask_prob', 'N/A')}</td>
            <td>{r['epochs']}</td>
            <td>{r['tcn_rmse']:.4f}</td>
            <td style="color: #7f8c8d;">{format_seconds(r['runtime_seconds'])}</td>
        </tr>
        """)

    html = f"""
    <!DOCTYPE html>
    <html lang="hu">
    <head>
        <meta charset="UTF-8">
        <title>Research Summary Results</title>
        <style>
            @media print {{
                .no-print {{ display: none; }}
                body {{ font-size: 10pt; background-color: white !important; margin: 0; }}
                .card {{ box-shadow: none !important; border: 1px solid #eee !important; }}
            }}
            
            body {{ 
                font-family: 'Segoe UI', Helvetica, Arial, sans-serif; 
                line-height: 1.4; 
                color: #2c3e50; 
                max-width: 1500px; 
                margin: 0 auto; 
                padding: 30px;
                background-color: #f4f7f9;
            }}
            
            .header-main {{
                text-align: center;
                border-bottom: 3px solid #2c3e50;
                margin-bottom: 25px;
                padding-bottom: 10px;
            }}
            
            .header-main h1 {{ 
                margin: 0; 
                text-transform: uppercase; 
                letter-spacing: 2px;
                font-size: 26px;
                color: #2c3e50;
            }}

            .grid-container {{
                display: grid;
                grid-template-columns: 1fr 1fr;
                gap: 20px;
                margin-bottom: 25px;
            }}
            
            .info-card {{
                background: white;
                padding: 18px;
                border-radius: 8px;
                box-shadow: 0 2px 4px rgba(0,0,0,0.04);
                border-top: 4px solid #34495e;
            }}

            .summary-card {{
                background: white;
                padding: 18px;
                border-radius: 8px;
                box-shadow: 0 2px 4px rgba(0,0,0,0.04);
                border-top: 4px solid #27ae60;
            }}
            
            .label {{ 
                font-weight: bold; 
                color: #7f8c8d; 
                text-transform: uppercase; 
                font-size: 10px; 
                display: block;
                margin-bottom: 2px;
            }}
            
            .value {{ font-size: 14px; margin-bottom: 8px; display: block; }}
            
            .card {{
                background: white;
                padding: 20px;
                border-radius: 8px;
                box-shadow: 0 2px 10px rgba(0,0,0,0.05);
                margin-bottom: 20px;
                overflow-x: auto;
            }}
            
            h2 {{ color: #2c3e50; border-left: 5px solid #27ae60; padding-left: 12px; margin-top: 10px; font-size: 20px; }}
            
            table {{ 
                width: 100%; 
                border-collapse: collapse; 
                margin-top: 15px;
                background: white;
            }}
            
            th, td {{ 
                padding: 10px 6px; 
                border: 1px solid #e0e0e0; 
                text-align: center; 
                font-size: 12px;
            }}
            
            th {{ 
                background-color: #f8f9fa; 
                color: #34495e;
                font-weight: bold;
                text-transform: uppercase;
                font-size: 10px;
                position: sticky;
                top: 0;
                z-index: 10;
            }}

            tr:nth-child(even) {{ background-color: #fafafa; }}
            tr:hover {{ background-color: #f1f8ff; }}

            .footer-text {{
                text-align: center;
                font-size: 11px;
                color: #bdc3c7;
                margin-top: 40px;
            }}

            code {{ background: #f1f3f5; padding: 2px 4px; border-radius: 3px; font-family: monospace; }}
        </style>
    </head>
    <body>

    <div class="header-main">
        <h1>--- Research Summary Results ---</h1>
        <p style="color: #7f8c8d; margin: 5px 0 0 0; font-size: 14px;">{project_name}</p>
    </div>

    <div class="grid-container">
        <div class="info-card">
            <span class="label">Researcher's name</span>
            <span class="value">{researcher_name}</span>
            
            <span class="label">Institution</span>
            <span class="value">{institution}</span>
            
            <span class="label">Project / Version</span>
            <span class="value">{project_name} <span style="color:#27ae60;">[v{version}]</span></span>
        </div>

        <div class="summary-card">
            <div style="display: grid; grid-template-columns: 1fr 1fr;">
                <div>
                    <span class="label">Number of tests run</span>
                    <span class="value" style="font-size: 22px; font-weight: bold;">{len(all_results)} piece</span>
                </div>
                <div>
                    <span class="label">Generation date</span>
                    <span class="value">{now}</span>
                </div>
            </div>
            <span class="label">Output folder</span>
            <span class="value" style="font-family: monospace; font-size: 11px; color: #7f8c8d;">{base_output_dir}</span>
            <span class="label">Comment:</span>
            <span class="value" style="font-family: monospace; font-size: 11px; color: #7f8c8d;">Extra_col contains any extra columns that are omitted from the masking beyond the target columns.</span>
        </div>
    </div>

    <div class="card">
        <h2>Comparison of Experimental Variations</h2>
        <table>
            <thead>
                <tr>
                    <th>Test name</th>
                    <th>Features</th>
                    <th>Target col</th>
                    <th>Extra col</th>
                    <th>In Window</th>
                    <th>Out Horizon</th>
                    <th>Mask %</th>
                    <th>Epochs</th>
                    <th>TCN RMSE</th>
                    <th>Runtime</th>
                </tr>
            </thead>
            <tbody>
                {''.join(rows)}
            </tbody>
        </table>
    </div>

    <div class="footer-text no-print">
        Document generated: {now} | {institution} | {researcher_name}
    </div>

    </body>
    </html>
    """
    
    with open(os.path.join(base_output_dir, "master_summary.html"), "w", encoding="utf-8") as f:
        f.write(html)

#DEF: format_second
#A másodpercben lévő időt konvertálom perc és másodpercre. Ez ekll hogy a riportba legalább percre tördelve legyen látható az idő.
def format_seconds(sec):
    m = int(sec // 60)
    s = int(sec % 60)
    return f"{m}m {s}s"


#CLASS: TimeSeriesDataset
class TimeSeriesDataset(Dataset):
    def __init__(self, data, input_window, output_horizon, target_idx, mask_prob=0.3, always_keep=[0, 1]):
        """
        data: A már skálázott numpy tömb (features + target)
        target_idx: Melyik oszlop a cél (pl. temp_c indexe)
        mask_prob: Milyen eséllyel maszkoljunk el paramétereket a tanítás során
        always_keep: Azok az indexek, amiket SOHA nem maszkolunk (pl. idő és temp_c)
        """
        self.data = torch.tensor(data, dtype=torch.float32)
        self.input_window = input_window
        self.output_horizon = output_horizon
        self.target_idx = target_idx
        self.mask_prob = mask_prob
        self.always_keep = always_keep

    #Visszaadja az adatszet hosszát
    def __len__(self):
        return len(self.data) - self.input_window - self.output_horizon

    #Ablakozó
    def __getitem__(self, idx):
        # Bemeneti ablak kivágása (aktuális kejzetünk-től a helyzetünk+az ablakméret)
        x = self.data[idx : idx + self.input_window].clone()
        
        # Cél ablak kivágása (az x végétől a predikciós ablak méretével)
        y = self.data[idx + self.input_window : idx + self.input_window + self.output_horizon, self.target_idx]

        # --- RANDOM MASKING ---
        # Csak tanítás közben érdemes, de nem elfelejtendő hogy test módban is emulálva lesz a hiány.
        if self.mask_prob > 0: #Ha nagyobb mint 0 tehát 0 esetén ki van kapcsolva, a filben 0-t kell használni ha nem kell.
            for feat_idx in range(x.shape[1]):
                if feat_idx not in self.always_keep:
                    if np.random.rand() < self.mask_prob:
                        x[:, feat_idx] = -1.0 # Maszkoló érték, mert az átskálázás miatt ezt normál esetben úgy sem veheti fel.
        
        return x, y

#CLASS: ResidualBlock
#A TCN modelhez kell összeállítani a rétegeket ez az osztály elvégzi és viszaadja hogy össze lehessen fűzni.
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super(ResidualBlock, self).__init__()
        padding = (kernel_size - 1) * dilation
        
        self.conv1 = parametrizations.weight_norm(nn.Conv1d(in_channels, out_channels, kernel_size,
                                               padding=padding, dilation=dilation))
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        
        self.conv2 = parametrizations.weight_norm(nn.Conv1d(out_channels, out_channels, kernel_size,
                                               padding=padding, dilation=dilation))
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        
        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1,
                                 self.conv2, self.chomp2, self.relu2)
        
        # Ha a bemeneti és kimeneti csatornaszám eltér, kell egy skip-connection illesztés
        self.downsample = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super(Chomp1d, self).__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TCNModel(nn.Module):
    def __init__(self, num_inputs, output_horizon, num_targets,
                 num_channels=[64, 64, 128], kernel_size=3): # num_channels=[64, 64, 128]
        super(TCNModel, self).__init__()

        self.num_targets = num_targets
        self.output_horizon = output_horizon

        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = num_inputs if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers.append(ResidualBlock(in_channels, out_channels, kernel_size, dilation=dilation_size))

        self.tcn = nn.Sequential(*layers)

        # Kimeneti réteg több targetre
        self.linear = nn.Linear(num_channels[-1], output_horizon * num_targets)

    def forward(self, x):
        # x: [Batch, Time, Features]
        x = x.transpose(1, 2)  # -> [Batch, Features, Time]

        y = self.tcn(x)        # -> [Batch, Channels, Time]

        out = self.linear(y[:, :, -1])  # -> [Batch, output_horizon * num_targets]

        # Visszaalakítás 3D formára
        out = out.view(-1, self.output_horizon, self.num_targets)
        return out

#DEF: calculate_metrics_overall
#Előállítom a négyzetes és abszolút hibát és a négyzetes gyökét. Ki is van íratva és vissza van adva a loghoz.
def calculate_metrics_overall(y_true, y_pred, model_name="Modell"):
    mse = mean_squared_error(y_true.reshape(-1), y_pred.reshape(-1))
    mae = mean_absolute_error(y_true.reshape(-1), y_pred.reshape(-1))
    rmse = np.sqrt(mse)

    print(f"--- {model_name} Összesített eredmények ---")
    print(f"MSE:  {mse:.4f}")
    print(f"MAE:  {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")

    return { "mse": float(mse), "mae": float(mae), "rmse": float(rmse) }

def calculate_metrics_overall_fixed(y_true, y_pred, model_name="TCN"):
    # y_true és y_pred formátuma: (N, H, T)
    N, H, T = y_true.shape
    
    # A meglévő függvényed kiszámolja horizontonként az átlagos MAE/RMSE értékeket
    # (Ezek már a teljes tesztidőszak átlagai az adott órára)
    mae_per_h, rmse_per_h = calculate_metrics_per_horizon(y_true, y_pred)
    
    # Az új, torzításmentes globális metrika: a horizontok hibáinak átlaga
    overall_mae = np.mean(mae_per_h)
    overall_rmse = np.mean(rmse_per_h)
    overall_mse = overall_rmse ** 2 # Visszafelé számolva a konzisztencia miatt
    
    print(f"\n--- {model_name} ÚJ, Torzításmentes Összesített eredmények ---")
    print(f"MSE:  {overall_mse:.4f}")
    print(f"MAE:  {overall_mae:.4f}")
    print(f"RMSE: {overall_rmse:.4f}")
    
    return { "mse": float(overall_mse), "mae": float(overall_mae), "rmse": float(overall_rmse) }

#DEF calculate_metrics_per_terget
#Ez a rsz lényeégben ugyan azt végzi amit az összesített metrika számoló de targetekre bontva. Tehát minden prediktált célváltozóra.
def calculate_metrics_per_target(y_true, y_pred, target_names):
    num_targets = y_true.shape[2]
    results = {}

    print("\n--- Targetenkénti eredmények ---")
    for t in range(num_targets):
        mse = mean_squared_error(y_true[:,:,t].reshape(-1), y_pred[:,:,t].reshape(-1))
        mae = mean_absolute_error(y_true[:,:,t].reshape(-1), y_pred[:,:,t].reshape(-1))
        rmse = np.sqrt(mse)

        name = target_names[t]
        print(f"{name}: MAE={mae:.4f}, RMSE={rmse:.4f}")

        results[name] = {"mae": float(mae), "rmse": float(rmse)}

    return results

#DEF: calculated_metrics_per_horizon
def calculate_metrics_per_horizon(y_true, y_pred):
    N, H, T = y_true.shape
    mae_per_h = []
    rmse_per_h = []

    for h in range(H):
        true_h = y_true[:, h, :].reshape(-1)
        pred_h = y_pred[:, h, :].reshape(-1)

        mae = mean_absolute_error(true_h, pred_h)
        rmse = np.sqrt(mean_squared_error(true_h, pred_h))

        mae_per_h.append(mae)
        rmse_per_h.append(rmse)

    return np.array(mae_per_h), np.array(rmse_per_h)

#DEF: calculate_metrics_per_horizon_per_target
def calculate_metrics_per_horizon_per_target(y_true, y_pred, target_names):
    N, H, T = y_true.shape
    results = {}

    for t in range(T):
        mae_list = []
        rmse_list = []

        for h in range(H):
            true_h = y_true[:, h, t]
            pred_h = y_pred[:, h, t]

            mae = mean_absolute_error(true_h, pred_h)
            rmse = np.sqrt(mean_squared_error(true_h, pred_h))

            mae_list.append(mae)
            rmse_list.append(rmse)

        results[target_names[t]] = {
            "mae": np.array(mae_list),
            "rmse": np.array(rmse_list)
        }

    return results

#DEF:inverse_transform_multitarget
# Inverz skálázás (vissza Celsiusba, vagy minden másba ami benne van)
# Mivel a scaler oszlopra (features) vár adatot, de nekünk csak a predikciónk van.
def inverse_transform_multitarget(y_pred, y_test, scaler, target_idx, num_features):
    """
    y_pred, y_test: (N, horizon, num_targets)
    scaler: A fenti GroupScaler objektum
    """
    y_pred_real = np.zeros_like(y_pred)
    y_test_real = np.zeros_like(y_test)
    
    # Az all_features listából tudjuk, melyik index melyik név
    target_names = [scaler.all_features[i] for i in target_idx]

    for t, col_name in enumerate(target_names):
        y_pred_real[:, :, t] = scaler.inverse_transform_single_col(y_pred[:, :, t], col_name)
        y_test_real[:, :, t] = scaler.inverse_transform_single_col(y_test[:, :, t], col_name)

    return y_pred_real, y_test_real

def save_plots(y_true, y_pred_tcn, target_names, out_dir, test_name):

    # TCN plot
    plt.figure(figsize=(15, 6))
    """plot_multitarget_predictions(
        y_true,
        y_pred_tcn,
        target_names,
        model_name="TCN"
    )"""
    plot_window_error_time_series(
        y_true,
        y_pred_tcn,
        target_names,
        model_name="TCN",
    )
    plt.savefig(os.path.join(out_dir, f"{test_name}_tcn_predictions.png"), dpi=150)
    plt.close()


def plot_multitarget_predictions(y_true, y_pred, target_names, model_name, samples=500):
    """
    y_true, y_pred: (N, H, T)
    samples: hány időlépést rajzoljunk ki (flattenelve)
    """
    N, H, T = y_true.shape

    # Laposítjuk idő szerint, hogy folytonos görbét kapjunk
    true_flat = y_true.reshape(-1, T)
    pred_flat = y_pred.reshape(-1, T)

    # Csak az első 'samples' pontot rajzoljuk
    true_flat = true_flat[:samples]
    pred_flat = pred_flat[:samples]

    fig, axes = plt.subplots(T, 1, figsize=(15, 4*T), sharex=True)

    if T == 1:
        axes = [axes]  # ha csak 1 target van, legyen lista

    for t in range(T):
        axes[t].plot(true_flat[:, t], label="Reality", color="black", linewidth=2, alpha=0.6)
        axes[t].plot(pred_flat[:, t], label=f"{model_name} prediction", linestyle="--")
        axes[t].set_title(f"{model_name} – {target_names[t]}")
        axes[t].set_ylabel(target_names[t])
        axes[t].grid(True, alpha=0.3)
        axes[t].legend()

    axes[-1].set_xlabel("Time step (hour)")
    plt.tight_layout()
   # plt.show()

def plot_window_error_time_series(y_true, y_pred, target_names, model_name):
    """
    y_true, y_pred formátuma: (N, H, T)
    N: egymást követő ablakok (időlépések) száma a tesztadatban
    H: kimeneti horizont (48 óra)
    T: targetek száma
    """
    N, H, T = y_true.shape
    
    # Kiszámoljuk az abszolút hibát minden pontra: (N, H, T)
    absolute_error = np.abs(y_pred - y_true)
    
    # Átlagolunk a horizont (H) mentén ablakonként.
    # Így minden ablakhoz (N) és minden targethez (T) kapunk egyetlen átlagos hibaértéket.
    # Formátum: (N, T)
    window_mae = np.mean(absolute_error, axis=1)
    
    # Grafikon létrehozása: minden target külön aldiagramot kap
    fig, axes = plt.subplots(T, 1, figsize=(15, 4 * T), sharex=True)
    
    if T == 1:
        axes = [axes]
        
    for t in range(T):
        # Kirajzoljuk az egymást követő ablakok hibáit
        axes[t].plot(window_mae[:, t], label=f"{model_name} Ablak Átlagos MAE", color="#e74c3c", linewidth=1.5)
        
        # Egy bázisvonal a 0 hibának (a tökéletes predikció)
        axes[t].axhline(0, color="black", linestyle="--", alpha=0.5)
        
        axes[t].set_title(f"{model_name} Előrejelzési Hiba az Időben – {target_names[t]}")
        axes[t].set_ylabel(f"Átlagos Abszolút Hiba ({target_names[t]})")
        axes[t].grid(True, alpha=0.3)
        axes[t].legend()
        
    axes[-1].set_xlabel("Egymást követő teszt ablakok (Óránkénti lépésekben)")
    plt.tight_layout()
    
    # Mentés a teszt könyvtárába
    #plt.savefig(os.path.join(out_dir, f"{test_name}_tcn_window_errors.png"), dpi=150)
    #plt.close()

def save_models_and_params(out_dir, model_tcn, scaler,
                           features, target_col, extra_cols,
                           input_window, output_horizon, mask_prob):

    torch.save(model_tcn.state_dict(), os.path.join(out_dir, "tcn_model.pth"))

    joblib.dump(scaler, os.path.join(out_dir, "scaler.pkl"))

    json.dump({
        "features": features,
        "target_col": target_col,
        "extra_cols": extra_cols,
        "input_window": input_window,
        "output_horizon": output_horizon,
        "mask_prob": mask_prob
    }, open(os.path.join(out_dir, "config.json"), "w"), indent=2)

def format_epoch_logs_table(epoch_logs, title):
    rows = []
    for e in epoch_logs:
        rows.append(f"""
        <tr>
            <td>{e['epoch']}</td>
            <td>{e['loss']:.6f}</td>
            <td>{e['duration']:.2f} s</td>
        </tr>
        """)
    return f"""
    <h3>{title}</h3>
    <table>
        <tr>
            <th>Epoch</th>
            <th>Loss</th>
            <th>Idő [s]</th>
        </tr>
        {''.join(rows)}
    </table>
    """

def format_horizon_table(mae_list, rmse_list, title):
    rows = []
    for h, (mae, rmse) in enumerate(zip(mae_list, rmse_list), start=1):
        rows.append(f"""
        <tr>
            <td>{h}</td>
            <td>{mae:.4f}</td>
            <td>{rmse:.4f}</td>
        </tr>
        """)

    return f"""
    <h3>{title}</h3>
    <table border="1" cellspacing="0" cellpadding="4">
        <tr>
            <th>Horizon</th>
            <th>MAE</th>
            <th>RMSE</th>
        </tr>
        {''.join(rows)}
    </table>
    """

def format_horizon_per_target_table(horizon_metrics, title):
    rows = []
    for target, metrics in horizon_metrics.items():
        mae_list = metrics["mae"]
        rmse_list = metrics["rmse"]
        for h, (mae, rmse) in enumerate(zip(mae_list, rmse_list), start=1):
            rows.append(f"""
            <tr>
                <td>{target}</td>
                <td>{h}</td>
                <td>{mae:.4f}</td>
                <td>{rmse:.4f}</td>
            </tr>
            """)

    return f"""
    <h3>{title}</h3>
    <table border="1" cellspacing="0" cellpadding="4">
        <tr>
            <th>Target</th>
            <th>Horizon</th>
            <th>MAE</th>
            <th>RMSE</th>
        </tr>
        {''.join(rows)}
    </table>
    """

#Tesztenkénti riport
def generate_html_report(out_dir, test_name,
                         tcn_overall, tcn_per_target,
                         features, target_col, extra_cols, input_window, output_horizon, mask_prob,
                         tcn_epoch_logs, epoch,
                         tcn_mae_h, tcn_rmse_h, tcn_horizon_metrics,
                         researcher_name="Bagi Béla", 
                         institution="University of Miskolc - Institute of Physics and Electrical Engineering",
                         project_name = "Home Weather Station Prediction",
                         version="1.0"):
    
    tcn_epoch_html = format_epoch_logs_table(tcn_epoch_logs, "TCN Epoch log")

    tcn_horizon_html = format_horizon_table(tcn_mae_h, tcn_rmse_h, "TCN Horizon metrics")

    tcn_horizon_target_html = format_horizon_per_target_table(tcn_horizon_metrics, "TCN Horizon per Target")
    
    now = datetime.now().strftime("%Y-%m-%d %H:%M")

    html = f"""
    <!DOCTYPE html>
    <html lang="hu">
    <head>
        <meta charset="UTF-8">
        <title>Research Report</title>
        <style>
            @media print {{
                .no-print {{ display: none; }}
                body {{ font-size: 11pt; background-color: white !important; margin: 0; }}
                .card {{ box-shadow: none !important; border: 1px solid #eee !important; page-break-inside: avoid; }}
            }}
            
            body {{ 
                font-family: 'Segoe UI', Helvetica, Arial, sans-serif; 
                line-height: 1.5; 
                color: #2c3e50; 
                max-width: 1100px; 
                margin: 0 auto; 
                padding: 40px;
                background-color: #f4f7f9;
            }}
            
            .header-main {{
                text-align: center;
                border-bottom: 3px solid #2c3e50;
                margin-bottom: 30px;
                padding-bottom: 10px;
            }}
            
            .header-main h1 {{ 
                margin: 0; 
                text-transform: uppercase; 
                letter-spacing: 2px;
                font-size: 28px;
            }}

            .grid-container {{
                display: grid;
                grid-template-columns: 1fr 1fr;
                gap: 20px;
                margin-bottom: 30px;
            }}
            
            .info-card {{
                background: white;
                padding: 20px;
                border-radius: 8px;
                box-shadow: 0 2px 4px rgba(0,0,0,0.04);
                border-top: 4px solid #34495e;
            }}

            .params-card {{
                background: white;
                padding: 20px;
                border-radius: 8px;
                box-shadow: 0 2px 4px rgba(0,0,0,0.04);
                border-top: 4px solid #3498db;
            }}
            
            .label {{ 
                font-weight: bold; 
                color: #7f8c8d; 
                text-transform: uppercase; 
                font-size: 10px; 
                display: block;
                margin-bottom: 2px;
            }}
            
            .value {{ font-size: 15px; margin-bottom: 10px; display: block; }}
            
            .card {{
                background: white;
                padding: 25px;
                border-radius: 8px;
                box-shadow: 0 2px 10px rgba(0,0,0,0.05);
                margin-bottom: 25px;
            }}
            
            h2 {{ color: #2c3e50; border-left: 5px solid #3498db; padding-left: 15px; margin-top: 40px; }}
            h3 {{ color: #2980b9; font-size: 18px; margin-top: 20px; border-bottom: 1px solid #eee; padding-bottom: 5px; }}
            
            pre {{ 
                background: #f8f9fa; 
                padding: 15px; 
                border-radius: 4px; 
                font-size: 12px;
                border: 1px solid #eaedf0;
                white-space: pre-wrap;
            }}
            
            .img-container {{ text-align: center; margin: 20px 0; }}
            .img-container img {{ 
                max-width: 100%; 
                border-radius: 4px; 
                border: 1px solid #ddd;
            }}

            .footer-text {{
                text-align: center;
                font-size: 12px;
                color: #bdc3c7;
                margin-top: 50px;
            }}
        </style>
    </head>
    <body>

    <div class="header-main">
        <h1>--- Research Report ---</h1>
        <p style="color: #7f8c8d; margin-top: 5px;">{project_name}</p>
    </div>

    <div class="grid-container">
        <div class="info-card">
            <span class="label">Researcher's name</span>
            <span class="value">{researcher_name}</span>
            
            <span class="label">Institution</span>
            <span class="value">{institution}</span>
            
            <span class="label">Project / Version</span>
            <span class="value">{project_name} <span style="color:#3498db;">[v{version}]</span></span>
        </div>

        <div class="params-card">
            <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 10px;">
                <div>
                    <span class="label">Report ID (Test)</span>
                    <span class="value" style="font-weight:bold;">{test_name}</span>
                    
                    <span class="label">Generation date</span>
                    <span class="value">{now}</span>
                </div>
                <div>
                    <span class="label">Epoch number</span>
                    <span class="value">{epoch}</span>

                    <span class="label">Mask Probability</span>
                    <span class="value">{mask_prob}</span>
                </div>
            </div>
            <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 10px; border-top: 1px solid #eee; pt: 10px; margin-top: 5px;">
                <div>
                    <span class="label">Input Window</span>
                    <span class="value">{input_window}</span>
                </div>
                <div>
                    <span class="label">Output Horizon</span>
                    <span class="value">{output_horizon}</span>
                </div>
            </div>
        </div>
    </div>

    <div class="card">
        <h3>Model configuration</h3>
        <span class="label">Features (Inputs)</span>
        <div style="font-size: 13px; background: #fdfdfd; padding: 10px; border: 1px solid #f0f0f0;">{features}</div>
        <div style="margin-top:15px;">
            <span class="label">Target Columns</span>
            <code>{target_col}</code>
        </div>
        <div style="margin-top:15px;">
            <span class="label">Extra Columns (Exceptions to masking.)</span>
            <code>{extra_cols}</code>
        </div>
    </div>

    <h2>Overall results</h2>
    <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px;">
        <div class="card">
            <h3>TCN Overall</h3>
            <pre>{tcn_overall}</pre>
        </div>
    </div>
    <h2>Detailed Target metrics</h2>
    <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px;">
    <div class="card">
        <h3>TCN (Target)</h3>
        <pre>{tcn_per_target}</pre>
    </div>
    </div>

    <h2>Detailed Horizon metrics</h2>
    <span class="label">Comment:</span>
    <span class="value" style="font-family: monospace; font-size: 11px; color: #7f8c8d;">It shows how accurate the model is for different forecast distances. (Horizon in hours)</span>
    <div style="display: flex; gap: 20px; overflow-x: auto;">
    <div class="card">
        {tcn_horizon_html}
    </div>
    <div class="card">
        {tcn_horizon_target_html}
    </div>
    </div>

    <h2>Predictive Visualization</h2>
    <div class="card">
        <h3>TCN Model results</h3>
        <div class="img-container">
            <img src="{test_name}_tcn_predictions.png" alt="TCN Predictions">
        </div>
    </div>
    

    <div style="page-break-before: always;">
        <h2>Teaching process (Logs)</h2>
        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px;">
        <div class="card" style="font-size: 12px;">
            {tcn_epoch_html}
        </div>
        </div>
    </div>

    <div class="footer-text no-print">
        Document generated: {now} | {institution} | {researcher_name}
    </div>

    </body>
    </html>
    """

    filename = os.path.join(out_dir, f"{test_name}_report.html")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(html)


# --- Konfigurációs futtatás indítása ---
#run_experiments("test_variation_list-test.tsv")
#run_experiments("test_variation_list.tsv")
#run_experiments("test_variation_list-1-1.tsv","experiments6")
#run_experiments("test_variation_list-1-2.tsv","experiments8")
#run_experiments("test_variation_list2.tsv")
#run_experiments("test_variation_list3.tsv")
#run_experiments("test_variation_list-1-1.tsv","experiments12")
run_experiments("test_variation_list-Final2.tsv","experiments997")




=== Futtatás: test_002 ===
Adatok betöltve: 754800 sor, 15 jellemző.
Tanító batch-ek száma: 11555
Tanítás megkezdése a(z) cuda eszközön...
TCN - Epoch 1 | Batch 0/11555 | Loss: 0.408060
TCN - Epoch 1 | Batch 400/11555 | Loss: 0.000183
TCN - Epoch 1 | Batch 800/11555 | Loss: 0.000174
TCN - Epoch 1 | Batch 1200/11555 | Loss: 0.000134
TCN - Epoch 1 | Batch 1600/11555 | Loss: 0.000123
TCN - Epoch 1 | Batch 2000/11555 | Loss: 0.000149
TCN - Epoch 1 | Batch 2400/11555 | Loss: 0.000117
TCN - Epoch 1 | Batch 2800/11555 | Loss: 0.000176
TCN - Epoch 1 | Batch 3200/11555 | Loss: 0.000079
TCN - Epoch 1 | Batch 3600/11555 | Loss: 0.000085
TCN - Epoch 1 | Batch 4000/11555 | Loss: 0.000109
TCN - Epoch 1 | Batch 4400/11555 | Loss: 0.000098
TCN - Epoch 1 | Batch 4800/11555 | Loss: 0.000145
TCN - Epoch 1 | Batch 5200/11555 | Loss: 0.000158
TCN - Epoch 1 | Batch 5600/11555 | Loss: 0.000082
TCN - Epoch 1 | Batch 6000/11555 | Loss: 0.000171
TCN - Epoch 1 | Batch 6400/11555 | Loss: 0.000123
TCN - Epoch 1 |

<Figure size 1500x600 with 0 Axes>